# Bridge Notebook 03 — FASTA I/O and Sequence Database APIs

## From sequence files to reusable datasets

Este notebook enseña cómo pasar desde secuencias biológicas almacenadas en archivos FASTA hacia un dataset exportable para análisis posteriores.

La lógica general será:

```text
sequence IDs
↓
API queries
↓
FASTA files
↓
metadata tables
↓
sequence descriptors
↓
CSV dataset for EDA
```

Este notebook está pensado como puente entre los notebooks de representación de secuencias y un análisis exploratorio completo de datos biológicos.

## Learning objectives

Al final de este notebook deberías poder:

1. Leer archivos FASTA con secuencias de DNA y proteínas.
2. Exportar secuencias a FASTA usando encabezados informativos.
3. Descargar secuencias y metadata desde UniProt usando su REST API.
4. Generar automáticamente una lista de aproximadamente 300 UniProt accessions.
5. Consultar otras bases de datos importantes usando APIs programáticas.
6. Construir un archivo CSV con IDs, organismos, secuencias y descriptores.
7. Preparar un dataset que pueda usarse después como input para una EDA completa.

## Key definitions

| Term | Definition |
|---|---|
| **Sequence** | Ordered biological string composed of DNA, RNA or amino-acid symbols. |
| **FASTA** | Plain-text format for storing biological sequences. Each record starts with a header line beginning with `>`, followed by one or more sequence lines. |
| **Accession** | Stable database identifier assigned to a biological record, for example a UniProt accession such as `P69905`. |
| **Entry name** | Human-readable database label associated with a record. In UniProt this is different from the accession. |
| **Organism** | Species or taxonomic source of the sequence. |
| **API** | Application Programming Interface. A structured way to request data from a database using code. |
| **Endpoint** | Specific URL exposed by an API for one type of operation, such as search or record retrieval. |
| **Metadata** | Information describing the sequence, such as organism, gene name, protein name, database source or sequence length. |
| **Descriptor** | Numerical or categorical value computed from a sequence, such as GC content, amino-acid entropy or hydrophobic fraction. |
| **Feature matrix** | Table where rows are samples/sequences and columns are descriptors/features. |
| **Provenance** | Information about where the data came from and how it was generated. |

## API sources used in this notebook

This notebook focuses on these resources:

| Database | Main use in this notebook | Example data type |
|---|---|---|
| **UniProtKB** | Protein sequences and protein metadata | Protein FASTA, organism, gene, protein name |
| **NCBI Entrez / E-utilities** | DNA/protein records from NCBI databases | Nucleotide or protein FASTA |
| **ENA Browser API** | DNA/RNA sequence retrieval by accession | Nucleotide FASTA |
| **RCSB PDB Data API** | Protein sequence and structure-linked metadata | Polymer sequence from PDB entries |

The primary dataset generated in this notebook will come from **UniProtKB**, because it is the cleanest starting point for collecting protein sequences with metadata.

## 0. Setup

The notebook is designed to work in two modes:

- **Offline mode**, which uses toy examples and can run without internet.
- **Online mode**, which performs real API calls and exports the real dataset.

To use the real APIs, set:

```python
RUN_ONLINE_API_CALLS = True
```

For NCBI, add your email in `NCBI_EMAIL`. This is good practice when using NCBI E-utilities.

In [1]:
from pathlib import Path
from collections import Counter
import io
import json
import math
import re
import textwrap
import time

import numpy as np
import pandas as pd

try:
    import requests
except ImportError:
    requests = None

RUN_ONLINE_API_CALLS = True  # Change to True when running with internet access.

OUTPUT_DIR = Path("../outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

NCBI_EMAIL = "krenai@umag.cl"  # Replace before running NCBI calls.
NCBI_API_KEY = None  # Optional. Leave as None if you do not have one.

print(f"Output directory: {OUTPUT_DIR.resolve()}")
print(f"Online API calls enabled: {RUN_ONLINE_API_CALLS}")

Output directory: /home/david/Desktop/depto_computacion/intro_2026/demos_intr_2026/outputs
Online API calls enabled: True


# 1. FASTA files

## 1.1 What is FASTA?

A FASTA file is one of the most common formats for biological sequences.

A FASTA record has two parts:

```text
>sequence_id optional description
ATGCGTACGTAGCTAGCTAG
```

The first line is the **header**. It starts with `>`.

The following lines contain the biological sequence.

In this course, we will use FASTA headers that include:

```text
>sequence_id|organism|molecule=protein|source=UniProtKB|description=...
```

This is not the only possible FASTA style, but it is convenient for teaching because it keeps the ID and organism visible.

In [2]:
def wrap_sequence(sequence, width=60):
    """Wrap a sequence into lines of fixed width for FASTA export."""
    sequence = str(sequence).replace("\n", "").replace(" ", "").upper()
    return "\n".join(sequence[i:i + width] for i in range(0, len(sequence), width))


def clean_header_value(value):
    """Remove characters that would make FASTA headers harder to parse."""
    value = str(value) if value is not None else ""
    value = value.replace("\n", " ").replace("\r", " ")
    value = value.replace("|", ";")
    value = re.sub(r"\s+", " ", value).strip()
    return value


def make_fasta_header(record):
    """
    Build a teaching-friendly FASTA header.

    Expected record keys:
    - sequence_id
    - organism
    - molecule_type
    - source
    - description
    """
    sequence_id = clean_header_value(record.get("sequence_id", "unknown_id"))
    organism = clean_header_value(record.get("organism", "Unknown organism"))
    molecule_type = clean_header_value(record.get("molecule_type", "unknown"))
    source = clean_header_value(record.get("source", "unknown"))
    description = clean_header_value(record.get("description", ""))

    header = f">{sequence_id}|{organism}|molecule={molecule_type}|source={source}"
    if description:
        header += f"|description={description}"
    return header


def write_fasta(records, path, width=60):
    """Write a list of sequence records to a FASTA file."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("w", encoding="utf-8") as handle:
        for record in records:
            sequence = str(record["sequence"]).replace("\n", "").replace(" ", "").upper()
            handle.write(make_fasta_header(record) + "\n")
            handle.write(wrap_sequence(sequence, width=width) + "\n")

    return path


def parse_fasta_header(header):
    """
    Parse the simple FASTA header style used in this notebook.

    This parser also works with many generic FASTA headers, but metadata extraction
    will be more limited if the header is not pipe-delimited.
    """
    header = header.lstrip(">").strip()
    parts = header.split("|")

    parsed = {
        "sequence_id": parts[0].strip() if parts else header,
        "organism": None,
        "molecule_type": None,
        "source": None,
        "description": None,
        "raw_header": header,
    }

    if len(parts) > 1 and "=" not in parts[1]:
        parsed["organism"] = parts[1].strip()

    for part in parts[2:]:
        if "=" in part:
            key, value = part.split("=", 1)
            key = key.strip()
            value = value.strip()
            if key == "molecule":
                parsed["molecule_type"] = value
            else:
                parsed[key] = value

    return parsed


def read_fasta(path):
    """Read a FASTA file into a list of sequence records."""
    path = Path(path)
    records = []
    current_header = None
    current_sequence_lines = []

    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue

            if line.startswith(">"):
                if current_header is not None:
                    metadata = parse_fasta_header(current_header)
                    metadata["sequence"] = "".join(current_sequence_lines).upper()
                    records.append(metadata)

                current_header = line
                current_sequence_lines = []
            else:
                current_sequence_lines.append(line)

    if current_header is not None:
        metadata = parse_fasta_header(current_header)
        metadata["sequence"] = "".join(current_sequence_lines).upper()
        records.append(metadata)

    return records

## 1.2 Local FASTA example

Before using APIs, we start with a tiny local example.

This is useful because the same functions will be used later for real database records.

In [3]:
toy_records = [
    {
        "sequence_id": "DNA_DEMO_001",
        "organism": "Synthetic construct",
        "molecule_type": "DNA",
        "source": "local_demo",
        "description": "Short synthetic DNA sequence",
        "sequence": "ATGCGTACGTAGCTAGCTAGCTAGCTA",
    },
    {
        "sequence_id": "DNA_DEMO_002",
        "organism": "Synthetic construct",
        "molecule_type": "DNA",
        "source": "local_demo",
        "description": "Second short synthetic DNA sequence",
        "sequence": "ATGCGTACGTAGCTAGATAGCTAGCTA",
    },
    {
        "sequence_id": "PROT_DEMO_001",
        "organism": "Synthetic construct",
        "molecule_type": "protein",
        "source": "local_demo",
        "description": "Short synthetic protein sequence",
        "sequence": "MKTFFVLLLCTFTVVNA",
    },
]

toy_fasta_path = write_fasta(toy_records, OUTPUT_DIR / "toy_sequences.fasta")
print(toy_fasta_path)
print(toy_fasta_path.read_text()[:500])

../outputs/toy_sequences.fasta
>DNA_DEMO_001|Synthetic construct|molecule=DNA|source=local_demo|description=Short synthetic DNA sequence
ATGCGTACGTAGCTAGCTAGCTAGCTA
>DNA_DEMO_002|Synthetic construct|molecule=DNA|source=local_demo|description=Second short synthetic DNA sequence
ATGCGTACGTAGCTAGATAGCTAGCTA
>PROT_DEMO_001|Synthetic construct|molecule=protein|source=local_demo|description=Short synthetic protein sequence
MKTFFVLLLCTFTVVNA



In [4]:
read_back_records = read_fasta(OUTPUT_DIR / "toy_sequences.fasta")
pd.DataFrame(read_back_records)

,sequence_id,organism,molecule_type,source,description,raw_header,sequence
0,DNA_DEMO_001,Synthetic construct,DNA,local_demo,Short synthetic DNA sequence,DNA_DEMO_001|Synthetic construct|molecule=DNA|...,ATGCGTACGTAGCTAGCTAGCTAGCTA
1,DNA_DEMO_002,Synthetic construct,DNA,local_demo,Second short synthetic DNA sequence,DNA_DEMO_002|Synthetic construct|molecule=DNA|...,ATGCGTACGTAGCTAGATAGCTAGCTA
2,PROT_DEMO_001,Synthetic construct,protein,local_demo,Short synthetic protein sequence,PROT_DEMO_001|Synthetic construct|molecule=pro...,MKTFFVLLLCTFTVVNA


# 2. Sequence descriptors

A **descriptor** is a numerical summary computed from a sequence.

For DNA, simple descriptors include:

- sequence length
- base frequencies
- GC content
- ambiguous base fraction

For proteins, simple descriptors include:

- sequence length
- amino-acid frequencies
- hydrophobic residue fraction
- charged residue fraction
- estimated molecular weight
- Shannon entropy
- k-mer entropy

These descriptors can become columns in a feature matrix.

## 2.1 Entropy-based descriptors

Entropy is a way to measure diversity or uncertainty in the symbols of a sequence.

For a sequence alphabet with symbol probabilities $p_i$, Shannon entropy is:

$$
H = -\sum_i p_i \log_2(p_i)
$$

Interpretation:

- Low entropy means the sequence is dominated by a small number of symbols.
- High entropy means the sequence uses symbols more evenly.

We will also calculate:

| Descriptor | Meaning |
|---|---|
| **normalized entropy** | Shannon entropy divided by the maximum possible entropy for the alphabet size. |
| **Simpson concentration** | Probability that two randomly selected symbols are the same. Higher values indicate dominance by fewer symbols. |
| **Gini impurity** | Probability that two randomly selected symbols are different. Higher values indicate greater diversity. |
| **k-mer entropy** | Entropy computed over short sequence words instead of single symbols. |

In [5]:
DNA_ALPHABET = set("ACGT")
RNA_ALPHABET = set("ACGU")
DNA_WITH_AMBIGUOUS = set("ACGTN")
PROTEIN_ALPHABET = set("ACDEFGHIKLMNPQRSTVWY")

HYDROPHOBIC_RESIDUES = set("AVILMFWY")
AROMATIC_RESIDUES = set("FWY")
POSITIVE_RESIDUES = set("KRH")
NEGATIVE_RESIDUES = set("DE")
CHARGED_RESIDUES = POSITIVE_RESIDUES | NEGATIVE_RESIDUES
POLAR_RESIDUES = set("STNQCY")

# Approximate residue masses in Daltons. These are residue masses used for simple teaching-level estimates.
AA_RESIDUE_MASS = {
    "A": 71.0788, "R": 156.1875, "N": 114.1038, "D": 115.0886,
    "C": 103.1388, "E": 129.1155, "Q": 128.1307, "G": 57.0519,
    "H": 137.1411, "I": 113.1594, "L": 113.1594, "K": 128.1741,
    "M": 131.1926, "F": 147.1766, "P": 97.1167, "S": 87.0782,
    "T": 101.1051, "W": 186.2132, "Y": 163.1760, "V": 99.1326,
}

WATER_MASS = 18.01528


def normalize_sequence(sequence):
    return str(sequence).replace("\n", "").replace(" ", "").upper()


def infer_molecule_type(sequence):
    """
    Infer molecule type from alphabet.

    Note: very short sequences can be ambiguous. If metadata already contains
    molecule type, prefer the metadata.
    """
    sequence = normalize_sequence(sequence)
    symbols = set(sequence)

    if symbols.issubset(DNA_WITH_AMBIGUOUS):
        return "DNA"
    if symbols.issubset(RNA_ALPHABET):
        return "RNA"
    if symbols.issubset(PROTEIN_ALPHABET):
        return "protein"
    return "unknown"


def symbol_frequencies(sequence, alphabet):
    sequence = normalize_sequence(sequence)
    counts = Counter(sequence)
    total = len(sequence)
    if total == 0:
        return {symbol: np.nan for symbol in alphabet}
    return {symbol: counts.get(symbol, 0) / total for symbol in alphabet}


def shannon_entropy_from_counts(counts):
    total = sum(counts.values())
    if total == 0:
        return np.nan

    entropy = 0.0
    for count in counts.values():
        if count > 0:
            p = count / total
            entropy -= p * math.log2(p)
    return entropy


def normalized_entropy(sequence, alphabet_size=None):
    sequence = normalize_sequence(sequence)
    counts = Counter(sequence)
    entropy = shannon_entropy_from_counts(counts)

    if alphabet_size is None:
        alphabet_size = len(counts)

    if alphabet_size <= 1 or pd.isna(entropy):
        return np.nan

    return entropy / math.log2(alphabet_size)


def simpson_concentration(sequence):
    sequence = normalize_sequence(sequence)
    counts = Counter(sequence)
    total = len(sequence)
    if total == 0:
        return np.nan
    return sum((count / total) ** 2 for count in counts.values())


def gini_impurity(sequence):
    value = simpson_concentration(sequence)
    if pd.isna(value):
        return np.nan
    return 1 - value


def get_kmers(sequence, k):
    sequence = normalize_sequence(sequence)
    if k <= 0:
        raise ValueError("k must be a positive integer")
    if len(sequence) < k:
        return []
    return [sequence[i:i + k] for i in range(len(sequence) - k + 1)]


def kmer_entropy(sequence, k=2):
    kmers = get_kmers(sequence, k=k)
    return shannon_entropy_from_counts(Counter(kmers))

In [6]:
def dna_descriptors(sequence):
    """Compute simple DNA descriptors."""
    sequence = normalize_sequence(sequence)
    counts = Counter(sequence)
    canonical_total = sum(counts.get(base, 0) for base in "ACGT")
    length = len(sequence)

    row = {
        "length": length,
        "valid_dna_acgtn": set(sequence).issubset(DNA_WITH_AMBIGUOUS),
        "gc_content": ((counts.get("G", 0) + counts.get("C", 0)) / canonical_total) if canonical_total else np.nan,
        "at_content": ((counts.get("A", 0) + counts.get("T", 0)) / canonical_total) if canonical_total else np.nan,
        "ambiguous_fraction": counts.get("N", 0) / length if length else np.nan,
        "shannon_entropy": shannon_entropy_from_counts(counts),
        "normalized_entropy": normalized_entropy(sequence, alphabet_size=4),
        "gini_impurity": gini_impurity(sequence),
        "simpson_concentration": simpson_concentration(sequence),
        "kmer_entropy_k2": kmer_entropy(sequence, k=2),
        "kmer_entropy_k3": kmer_entropy(sequence, k=3),
    }

    for base in "ACGTN":
        row[f"{base}_count"] = counts.get(base, 0)
        row[f"{base}_freq"] = counts.get(base, 0) / length if length else np.nan

    return row


def protein_descriptors(sequence):
    """Compute simple teaching-level protein descriptors."""
    sequence = normalize_sequence(sequence)
    counts = Counter(sequence)
    length = len(sequence)

    valid_residues = [aa for aa in sequence if aa in PROTEIN_ALPHABET]
    valid_length = len(valid_residues)

    estimated_mw = (
        sum(AA_RESIDUE_MASS.get(aa, 0.0) for aa in valid_residues) + WATER_MASS
        if valid_length > 0 else np.nan
    )

    row = {
        "length": length,
        "valid_protein_standard_20aa": set(sequence).issubset(PROTEIN_ALPHABET),
        "unknown_or_nonstandard_fraction": 1 - (valid_length / length) if length else np.nan,
        "estimated_molecular_weight_da": estimated_mw,
        "hydrophobic_fraction": sum(aa in HYDROPHOBIC_RESIDUES for aa in sequence) / length if length else np.nan,
        "aromatic_fraction": sum(aa in AROMATIC_RESIDUES for aa in sequence) / length if length else np.nan,
        "positive_fraction": sum(aa in POSITIVE_RESIDUES for aa in sequence) / length if length else np.nan,
        "negative_fraction": sum(aa in NEGATIVE_RESIDUES for aa in sequence) / length if length else np.nan,
        "charged_fraction": sum(aa in CHARGED_RESIDUES for aa in sequence) / length if length else np.nan,
        "polar_fraction": sum(aa in POLAR_RESIDUES for aa in sequence) / length if length else np.nan,
        "shannon_entropy": shannon_entropy_from_counts(counts),
        "normalized_entropy": normalized_entropy(sequence, alphabet_size=20),
        "gini_impurity": gini_impurity(sequence),
        "simpson_concentration": simpson_concentration(sequence),
        "max_residue_frequency": max((count / length for count in counts.values()), default=np.nan),
        "kmer_entropy_k2": kmer_entropy(sequence, k=2),
        "kmer_entropy_k3": kmer_entropy(sequence, k=3),
    }

    for aa in sorted(PROTEIN_ALPHABET):
        row[f"aa_{aa}_count"] = counts.get(aa, 0)
        row[f"aa_{aa}_freq"] = counts.get(aa, 0) / length if length else np.nan

    return row


def sequence_descriptors(record):
    """Compute descriptors for a FASTA/API record."""
    sequence = normalize_sequence(record.get("sequence", ""))
    molecule_type = record.get("molecule_type") or infer_molecule_type(sequence)
    molecule_type = str(molecule_type).lower()

    base_row = {
        "sequence_id": record.get("sequence_id"),
        "organism": record.get("organism"),
        "molecule_type": molecule_type,
        "source": record.get("source"),
        "description": record.get("description"),
        "sequence": sequence,
    }

    if molecule_type in {"dna", "nucleotide"}:
        desc = dna_descriptors(sequence)
    elif molecule_type == "rna":
        # RNA is converted to DNA-like representation only for simple nucleotide descriptors.
        desc = dna_descriptors(sequence.replace("U", "T"))
        desc["valid_rna_acgun"] = set(sequence).issubset(set("ACGUN"))
    elif molecule_type == "protein":
        desc = protein_descriptors(sequence)
    else:
        desc = {"length": len(sequence)}

    return {**base_row, **desc}


def build_descriptor_table(records):
    return pd.DataFrame([sequence_descriptors(record) for record in records])

## 2.2 Build descriptors from local FASTA records

This shows the same workflow we will use later for downloaded sequences.

In [7]:
toy_descriptor_table = build_descriptor_table(read_back_records)
toy_descriptor_table.head()

,sequence_id,organism,molecule_type,source,description,sequence,length,valid_dna_acgtn,gc_content,at_content,...,aa_S_count,aa_S_freq,aa_T_count,aa_T_freq,aa_V_count,aa_V_freq,aa_W_count,aa_W_freq,aa_Y_count,aa_Y_freq
0,DNA_DEMO_001,Synthetic construct,dna,local_demo,Short synthetic DNA sequence,ATGCGTACGTAGCTAGCTAGCTAGCTA,27,True,0.481481,0.518519,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,DNA_DEMO_002,Synthetic construct,dna,local_demo,Second short synthetic DNA sequence,ATGCGTACGTAGCTAGATAGCTAGCTA,27,True,0.444444,0.555556,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,PROT_DEMO_001,Synthetic construct,protein,local_demo,Short synthetic protein sequence,MKTFFVLLLCTFTVVNA,17,NaN,NaN,NaN,...,0.0,0.0,3.0,0.176471,3.0,0.176471,0.0,0.0,0.0,0.0


In [8]:
toy_descriptor_path = OUTPUT_DIR / "toy_sequence_descriptors.csv"
toy_descriptor_table.to_csv(toy_descriptor_path, index=False)
print(toy_descriptor_path)

../outputs/toy_sequence_descriptors.csv


# 3. UniProt API: collect protein sequences and metadata

UniProtKB is the main database we will use here for proteins.

The goal is to build a dataset with approximately **300 reviewed protein sequences**.

Default query:

```text
(reviewed:true) AND (organism_id:9606)
```

This means:

- `reviewed:true`: use reviewed UniProtKB/Swiss-Prot records.
- `organism_id:9606`: restrict to *Homo sapiens*.

You can replace `9606` with another NCBI taxonomy ID.

## 3.1 UniProt fields

We will request these fields:

| Field requested | Typical output column | Meaning |
|---|---|---|
| `accession` | Entry | Stable UniProt accession |
| `id` | Entry Name | UniProt entry name |
| `protein_name` | Protein names | Recommended/submitted protein names |
| `gene_names` | Gene Names | Associated gene names |
| `organism_name` | Organism | Species name |
| `organism_id` | Organism (ID) | NCBI taxonomy ID |
| `length` | Length | Sequence length |
| `sequence` | Sequence | Amino-acid sequence |
| `reviewed` | Reviewed | Reviewed/unreviewed status |

In [9]:
def require_requests():
    if requests is None:
        raise ImportError("The 'requests' package is required for API calls. Install it with: pip install requests")


def fetch_uniprot_tsv(query, fields, size=300, timeout=60):
    """
    Fetch UniProtKB records using the UniProt REST search endpoint.

    Parameters
    ----------
    query : str
        UniProt query string.
    fields : list[str]
        UniProt field names to request.
    size : int
        Number of records to request.
    timeout : int
        Request timeout in seconds.

    Returns
    -------
    pandas.DataFrame
        Table returned by UniProt.
    """
    require_requests()

    url = "https://rest.uniprot.org/uniprotkb/search"
    params = {
        "query": query,
        "fields": ",".join(fields),
        "format": "tsv",
        "size": size,
    }

    response = requests.get(
        url,
        params=params,
        timeout=timeout,
        headers={"User-Agent": "intro-bioinformatics-course-notebook/1.0"},
    )
    response.raise_for_status()

    return pd.read_csv(io.StringIO(response.text), sep="\t")


UNIPROT_FIELDS = [
    "accession",
    "id",
    "protein_name",
    "gene_names",
    "organism_name",
    "organism_id",
    "length",
    "sequence",
    "reviewed",
]

UNIPROT_QUERY = "(reviewed:true) AND (organism_id:9606)"
TARGET_UNIPROT_N = 300

print("UniProt query:", UNIPROT_QUERY)
print("Target number of UniProt records:", TARGET_UNIPROT_N)

UniProt query: (reviewed:true) AND (organism_id:9606)
Target number of UniProt records: 300


## 3.2 Generate a list of approximately 300 UniProt IDs

This cell creates the requested list of UniProt accessions.

When online calls are enabled, the output file will be:

```text
outputs/uniprot_accessions_300.txt
```

In [10]:
if RUN_ONLINE_API_CALLS:
    uniprot_raw = fetch_uniprot_tsv(
        query=UNIPROT_QUERY,
        fields=UNIPROT_FIELDS,
        size=TARGET_UNIPROT_N,
    )

    uniprot_ids_300 = uniprot_raw["Entry"].dropna().astype(str).tolist()

    accessions_path = OUTPUT_DIR / "uniprot_accessions_300.txt"
    accessions_path.write_text("\n".join(uniprot_ids_300) + "\n", encoding="utf-8")

    print(f"Retrieved {len(uniprot_ids_300)} UniProt accessions")
    print(f"Saved to: {accessions_path}")
    print(uniprot_ids_300[:20])
else:
    uniprot_raw = pd.DataFrame()
    uniprot_ids_300 = []
    print("Online UniProt call skipped.")
    print("Set RUN_ONLINE_API_CALLS = True to create outputs/uniprot_accessions_300.txt")

Retrieved 300 UniProt accessions
Saved to: ../outputs/uniprot_accessions_300.txt
['A0A0C5B5G6', 'A0A1B0GTW7', 'A0JNW5', 'A0JP26', 'A0PK11', 'A1A4S6', 'A1A519', 'A1L190', 'A1L3X0', 'A1X283', 'A2A2Y4', 'A2RU14', 'A2RUB6', 'A2RUC4', 'A3KN83', 'A4D1B5', 'A4GXA9', 'A5D8V7', 'A5PLL7', 'A6BM72']


## 3.3 Standardize UniProt metadata

UniProt column names are useful but not always convenient for downstream analysis.

We will standardize them to snake_case names.

In [11]:
def standardize_uniprot_table(df):
    """Standardize common UniProt TSV columns."""
    if df.empty:
        return df.copy()

    column_map = {
        "Entry": "uniprot_accession",
        "Entry Name": "uniprot_entry_name",
        "Protein names": "protein_name",
        "Gene Names": "gene_names",
        "Organism": "organism",
        "Organism (ID)": "organism_id",
        "Length": "length_uniprot",
        "Sequence": "sequence",
        "Reviewed": "reviewed",
    }

    standardized = df.rename(columns={k: v for k, v in column_map.items() if k in df.columns}).copy()
    return standardized


uniprot_table = standardize_uniprot_table(uniprot_raw)
uniprot_table.head()

,uniprot_accession,uniprot_entry_name,protein_name,gene_names,organism,organism_id,length_uniprot,sequence,reviewed
0,A0A0C5B5G6,MOTSC_HUMAN,Mitochondrial-derived peptide MOTS-c (Mitochon...,MT-RNR1,Homo sapiens (Human),9606,16,MRWQEMGYIFYPRKLR,reviewed
1,A0A1B0GTW7,CIROP_HUMAN,Ciliated left-right organizer metallopeptidase...,CIROP LMLN2,Homo sapiens (Human),9606,788,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,reviewed
2,A0JNW5,BLT3B_HUMAN,Bridge-like lipid transfer protein family memb...,BLTP3B KIAA0701 SHIP164 UHRF1BP1L,Homo sapiens (Human),9606,1464,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,reviewed
3,A0JP26,POTB3_HUMAN,POTE ankyrin domain family member B3,POTEB3,Homo sapiens (Human),9606,581,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,reviewed
4,A0PK11,CLRN2_HUMAN,Clarin-2,CLRN2,Homo sapiens (Human),9606,232,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,reviewed


## 3.4 Convert UniProt records into FASTA records

We now transform the UniProt table into the same record structure used by the FASTA writer.

In [12]:
def uniprot_table_to_records(df):
    records = []

    if df.empty:
        return records

    for _, row in df.iterrows():
        accession = row.get("uniprot_accession")
        organism = row.get("organism")
        protein_name = row.get("protein_name")
        gene_names = row.get("gene_names")
        entry_name = row.get("uniprot_entry_name")

        description_parts = []
        if pd.notna(entry_name):
            description_parts.append(f"entry_name={entry_name}")
        if pd.notna(protein_name):
            description_parts.append(f"protein_name={protein_name}")
        if pd.notna(gene_names):
            description_parts.append(f"genes={gene_names}")

        records.append({
            "sequence_id": accession,
            "organism": organism,
            "molecule_type": "protein",
            "source": "UniProtKB",
            "description": "; ".join(description_parts),
            "sequence": row.get("sequence"),
        })

    return records


uniprot_records = uniprot_table_to_records(uniprot_table)
print(f"UniProt FASTA records prepared: {len(uniprot_records)}")

UniProt FASTA records prepared: 300


## 3.5 Export UniProt FASTA, metadata CSV and descriptor CSV

The main files generated by this section are:

```text
outputs/uniprot_proteins_300.fasta
outputs/uniprot_proteins_300_metadata.csv
outputs/uniprot_proteins_300_descriptors.csv
```

In [13]:
if RUN_ONLINE_API_CALLS and len(uniprot_records) > 0:
    uniprot_fasta_path = write_fasta(uniprot_records, OUTPUT_DIR / "uniprot_proteins_300.fasta")
    uniprot_metadata_path = OUTPUT_DIR / "uniprot_proteins_300_metadata.csv"
    uniprot_descriptors_path = OUTPUT_DIR / "uniprot_proteins_300_descriptors.csv"

    uniprot_table.to_csv(uniprot_metadata_path, index=False)

    uniprot_descriptor_table = build_descriptor_table(uniprot_records)
    uniprot_descriptor_table.to_csv(uniprot_descriptors_path, index=False)

    print("Saved:")
    print("-", uniprot_fasta_path)
    print("-", uniprot_metadata_path)
    print("-", uniprot_descriptors_path)
else:
    print("UniProt export skipped in offline mode.")

Saved:
- ../outputs/uniprot_proteins_300.fasta
- ../outputs/uniprot_proteins_300_metadata.csv
- ../outputs/uniprot_proteins_300_descriptors.csv


## 3.6 Fetch UniProt records from an explicit list of accessions

Sometimes you do not want the first 300 records from a query. Instead, you may already have a curated list of UniProt IDs.

This function fetches records using a list like:

```python
my_ids = ["P69905", "P68871", "P04637"]
```

In [14]:
def chunk_list(values, chunk_size):
    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]


def fetch_uniprot_by_accessions(accessions, fields=None, batch_size=50, sleep_seconds=0.2):
    """Fetch UniProt records for an explicit list of accessions."""
    if fields is None:
        fields = UNIPROT_FIELDS

    frames = []

    for batch in chunk_list(list(accessions), batch_size):
        query = " OR ".join(f"accession:{accession}" for accession in batch)
        query = f"({query})"
        frame = fetch_uniprot_tsv(query=query, fields=fields, size=len(batch))
        frames.append(frame)
        time.sleep(sleep_seconds)

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


example_uniprot_ids = ["P69905", "P68871", "P04637", "P00533", "P01308"]

if RUN_ONLINE_API_CALLS:
    example_uniprot_df = fetch_uniprot_by_accessions(example_uniprot_ids)
    display(standardize_uniprot_table(example_uniprot_df).head())
else:
    print("Explicit UniProt accession fetch skipped in offline mode.")

,uniprot_accession,uniprot_entry_name,protein_name,gene_names,organism,organism_id,length_uniprot,sequence,reviewed
0,P01308,INS_HUMAN,Insulin [Cleaved into: Insulin B chain; Insuli...,INS,Homo sapiens (Human),9606,110,MALWMRLLPLLALLALWGPDPAAAFVNQHLCGSHLVEALYLVCGER...,reviewed
1,P68871,HBB_HUMAN,Hemoglobin subunit beta (Beta-globin) (Hemoglo...,HBB,Homo sapiens (Human),9606,147,MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESF...,reviewed
2,P69905,HBA_HUMAN,Hemoglobin subunit alpha (Alpha-globin) (Hemog...,HBA1; HBA2,Homo sapiens (Human),9606,142,MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPH...,reviewed
3,P00533,EGFR_HUMAN,Epidermal growth factor receptor (EC 2.7.10.1)...,EGFR ERBB ERBB1 HER1,Homo sapiens (Human),9606,1210,MRPSGTAGAALLALLAALCPASRALEEKKVCQGTSNKLTQLGTFED...,reviewed
4,P04637,P53_HUMAN,Cellular tumor antigen p53 (Antigen NY-CO-13) ...,TP53 P53,Homo sapiens (Human),9606,393,MEEPQSDPSVEPPLSQETFSDLWKLLPENNVLSPLPSQAMDDLMLS...,reviewed


# 4. NCBI E-utilities: retrieve DNA or protein FASTA

NCBI E-utilities provide programmatic access to Entrez databases.

For sequence work, common databases include:

| Database | Use |
|---|---|
| `nuccore` | nucleotide sequences |
| `protein` | protein sequences |
| `gene` | gene records |

Typical workflow:

```text
ESearch → get record IDs
EFetch  → download records in FASTA format
```

For teaching, we will use this workflow to retrieve DNA sequences from `nuccore`.

In [15]:
NCBI_EUTILS_BASE = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils"


def ncbi_esearch(db, term, retmax=10, email=None, api_key=None, timeout=60):
    """Search an NCBI Entrez database and return a list of IDs."""
    require_requests()

    params = {
        "db": db,
        "term": term,
        "retmax": retmax,
        "retmode": "json",
        "tool": "intro_bioinformatics_course_notebook",
    }

    if email:
        params["email"] = email
    if api_key:
        params["api_key"] = api_key

    response = requests.get(f"{NCBI_EUTILS_BASE}/esearch.fcgi", params=params, timeout=timeout)
    response.raise_for_status()
    data = response.json()

    return data.get("esearchresult", {}).get("idlist", [])


def ncbi_efetch_fasta(db, ids, rettype="fasta", email=None, api_key=None, timeout=60):
    """Fetch NCBI records in FASTA format."""
    require_requests()

    if isinstance(ids, str):
        ids = [ids]

    params = {
        "db": db,
        "id": ",".join(ids),
        "rettype": rettype,
        "retmode": "text",
        "tool": "intro_bioinformatics_course_notebook",
    }

    if email:
        params["email"] = email
    if api_key:
        params["api_key"] = api_key

    response = requests.get(f"{NCBI_EUTILS_BASE}/efetch.fcgi", params=params, timeout=timeout)
    response.raise_for_status()
    return response.text


# Example DNA query. You can replace this with another gene/organism combination.
NCBI_DNA_QUERY = "COX1[Gene] AND Homo sapiens[Organism]"

if RUN_ONLINE_API_CALLS:
    ncbi_ids = ncbi_esearch(
        db="nuccore",
        term=NCBI_DNA_QUERY,
        retmax=5,
        email=NCBI_EMAIL,
        api_key=NCBI_API_KEY,
    )
    print("NCBI IDs:", ncbi_ids)

    ncbi_fasta_text = ncbi_efetch_fasta(
        db="nuccore",
        ids=ncbi_ids,
        rettype="fasta",
        email=NCBI_EMAIL,
        api_key=NCBI_API_KEY,
    )

    ncbi_fasta_path = OUTPUT_DIR / "ncbi_dna_sequences.fasta"
    ncbi_fasta_path.write_text(ncbi_fasta_text, encoding="utf-8")
    print(f"Saved: {ncbi_fasta_path}")
else:
    print("NCBI API call skipped in offline mode.")

NCBI IDs: ['3347720687', '3349523114', '3349228795', '3349227439', '3347720882']
Saved: ../outputs/ncbi_dna_sequences.fasta


# 5. ENA Browser API: retrieve nucleotide FASTA by accession

The ENA Browser API is useful for retrieving European Nucleotide Archive records.

A simple FASTA retrieval pattern is:

```text
https://www.ebi.ac.uk/ena/browser/api/fasta/{accession}
```

This is convenient when you already know an accession.

In [16]:
def fetch_ena_fasta(accession, timeout=60):
    """Fetch a FASTA record from ENA by accession."""
    require_requests()
    url = f"https://www.ebi.ac.uk/ena/browser/api/fasta/{accession}"
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.text


# Example accession. Replace with a record relevant to your class or project.
ENA_EXAMPLE_ACCESSION = "X56734"

if RUN_ONLINE_API_CALLS:
    ena_fasta_text = fetch_ena_fasta(ENA_EXAMPLE_ACCESSION)
    ena_fasta_path = OUTPUT_DIR / f"ena_{ENA_EXAMPLE_ACCESSION}.fasta"
    ena_fasta_path.write_text(ena_fasta_text, encoding="utf-8")
    print(f"Saved: {ena_fasta_path}")
else:
    print("ENA API call skipped in offline mode.")

Saved: ../outputs/ena_X56734.fasta


# 6. RCSB PDB Data API: retrieve structure-linked protein sequence metadata

The PDB is not primarily a sequence database, but it is important because it connects sequences to experimentally determined or computed structures.

For a PDB entry, polymer entities contain sequence-related information.

Example pattern:

```text
https://data.rcsb.org/rest/v1/core/polymer_entity/{pdb_id}/{entity_id}
```

This is useful later when sequence EDA is connected with structural biology.

In [17]:
def fetch_rcsb_polymer_entity(entry_id, entity_id="1", timeout=60):
    """Fetch polymer entity metadata from the RCSB PDB Data API."""
    require_requests()
    url = f"https://data.rcsb.org/rest/v1/core/polymer_entity/{entry_id.upper()}/{entity_id}"
    response = requests.get(url, timeout=timeout)
    response.raise_for_status()
    return response.json()


def rcsb_polymer_entity_to_record(data, entry_id, entity_id="1"):
    """Convert an RCSB polymer entity JSON object into a sequence record."""
    entity_poly = data.get("entity_poly", {})
    identifiers = data.get("rcsb_polymer_entity_container_identifiers", {})
    source_organisms = data.get("rcsb_entity_source_organism", []) or []

    sequence = entity_poly.get("pdbx_seq_one_letter_code_can")
    uniprot_ids = identifiers.get("uniprot_ids", []) or []

    organism = "Unknown organism"
    if source_organisms:
        organism = source_organisms[0].get("scientific_name", organism)

    return {
        "sequence_id": f"{entry_id.upper()}_entity_{entity_id}",
        "organism": organism,
        "molecule_type": "protein",
        "source": "RCSB_PDB",
        "description": f"linked_uniprot_ids={','.join(uniprot_ids)}",
        "sequence": sequence,
    }


if RUN_ONLINE_API_CALLS:
    pdb_data = fetch_rcsb_polymer_entity("1A3N", "1")
    pdb_record = rcsb_polymer_entity_to_record(pdb_data, "1A3N", "1")
    display(pd.DataFrame([pdb_record]))
else:
    print("RCSB PDB API call skipped in offline mode.")

,sequence_id,organism,molecule_type,source,description,sequence
0,1A3N_entity_1,Homo sapiens,protein,RCSB_PDB,linked_uniprot_ids=P69905,VLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHF...


# 7. Combine sequences into one reusable dataset

For a later EDA notebook, we want a single CSV file with:

- sequence ID
- database source
- molecule type
- organism
- description
- sequence
- descriptors

In a real run, the main source will be UniProt. DNA records from NCBI/ENA can be added when needed.

In [18]:
# Offline demonstration using local records.
combined_records = list(toy_records)

# In online mode, you could replace or extend the combined records like this:
if RUN_ONLINE_API_CALLS and len(uniprot_records) > 0:
    combined_records = list(uniprot_records)

combined_fasta_path = write_fasta(combined_records, OUTPUT_DIR / "combined_sequences.fasta")
combined_dataset = build_descriptor_table(combined_records)
combined_csv_path = OUTPUT_DIR / "combined_sequence_dataset.csv"
combined_dataset.to_csv(combined_csv_path, index=False)

print("Saved:")
print("-", combined_fasta_path)
print("-", combined_csv_path)

combined_dataset.head()

Saved:
- ../outputs/combined_sequences.fasta
- ../outputs/combined_sequence_dataset.csv


,sequence_id,organism,molecule_type,source,description,sequence,length,valid_protein_standard_20aa,unknown_or_nonstandard_fraction,estimated_molecular_weight_da,...,aa_S_count,aa_S_freq,aa_T_count,aa_T_freq,aa_V_count,aa_V_freq,aa_W_count,aa_W_freq,aa_Y_count,aa_Y_freq
0,A0A0C5B5G6,Homo sapiens (Human),protein,UniProtKB,entry_name=MOTSC_HUMAN; protein_name=Mitochond...,MRWQEMGYIFYPRKLR,16,True,0.0,2174.61248,...,0,0.000000,0,0.000000,0,0.000000,1,0.062500,2,0.125000
1,A0A1B0GTW7,Homo sapiens (Human),protein,UniProtKB,entry_name=CIROP_HUMAN; protein_name=Ciliated ...,MLLLLLLLLLLPPLVLRVAASRCLHDETQKSVSLLRPPFSQLPSKS...,788,True,0.0,85396.90478,...,76,0.096447,48,0.060914,42,0.053299,13,0.016497,20,0.025381
2,A0JNW5,Homo sapiens (Human),protein,UniProtKB,entry_name=BLT3B_HUMAN; protein_name=Bridge-li...,MAGIIKKQILKHLSRFTKNLSPDKINLSTLKGEGELKNLELDEEVL...,1464,True,0.0,164198.50728,...,177,0.120902,85,0.058060,82,0.056011,12,0.008197,35,0.023907
3,A0JP26,Homo sapiens (Human),protein,UniProtKB,entry_name=POTB3_HUMAN; protein_name=POTE anky...,MVAEVCSMPAASAVKKPFDLRSKMGKWCHHRFPCCRGSGKSNMGTS...,581,True,0.0,65710.34248,...,47,0.080895,22,0.037866,27,0.046472,5,0.008606,10,0.017212
4,A0PK11,Homo sapiens (Human),protein,UniProtKB,entry_name=CLRN2_HUMAN; protein_name=Clarin-2;...,MPGWFKKAWYGLASLLSFSSFILIIVALVVPHWLSGKILCQTGVDL...,232,True,0.0,25446.15678,...,13,0.056034,7,0.030172,25,0.107759,5,0.021552,6,0.025862


## 7.1 Recommended output files for the next EDA notebook

After running this notebook online, the files most useful for downstream EDA are:

```text
outputs/uniprot_accessions_300.txt
outputs/uniprot_proteins_300.fasta
outputs/uniprot_proteins_300_metadata.csv
outputs/uniprot_proteins_300_descriptors.csv
outputs/combined_sequences.fasta
outputs/combined_sequence_dataset.csv
```

For the next notebook, the recommended input is:

```text
outputs/combined_sequence_dataset.csv
```

or, if focusing only on UniProt proteins:

```text
outputs/uniprot_proteins_300_descriptors.csv
```

# 8. Full UniProt pipeline function

This helper wraps the main UniProt workflow into one function.

Use it when you want to quickly regenerate the dataset.

In [19]:
def build_uniprot_protein_dataset(
    query="(reviewed:true) AND (organism_id:9606)",
    n_records=300,
    output_dir=OUTPUT_DIR,
    prefix="uniprot_proteins_300",
):
    """
    Build a protein dataset from UniProtKB.

    Outputs
    -------
    - accessions TXT
    - protein FASTA
    - metadata CSV
    - descriptors CSV
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    raw = fetch_uniprot_tsv(query=query, fields=UNIPROT_FIELDS, size=n_records)
    table = standardize_uniprot_table(raw)
    records = uniprot_table_to_records(table)
    descriptors = build_descriptor_table(records)

    accession_col = "uniprot_accession"
    accessions = table[accession_col].dropna().astype(str).tolist() if accession_col in table.columns else []

    paths = {
        "accessions": output_dir / f"{prefix}_accessions.txt",
        "fasta": output_dir / f"{prefix}.fasta",
        "metadata_csv": output_dir / f"{prefix}_metadata.csv",
        "descriptors_csv": output_dir / f"{prefix}_descriptors.csv",
    }

    paths["accessions"].write_text("\n".join(accessions) + "\n", encoding="utf-8")
    write_fasta(records, paths["fasta"])
    table.to_csv(paths["metadata_csv"], index=False)
    descriptors.to_csv(paths["descriptors_csv"], index=False)

    return {
        "raw": raw,
        "metadata": table,
        "records": records,
        "descriptors": descriptors,
        "paths": paths,
    }


if RUN_ONLINE_API_CALLS:
    dataset_result = build_uniprot_protein_dataset(
        query=UNIPROT_QUERY,
        n_records=TARGET_UNIPROT_N,
        output_dir=OUTPUT_DIR,
        prefix="uniprot_proteins_300",
    )
    print(json.dumps({k: str(v) for k, v in dataset_result["paths"].items()}, indent=2))
else:
    print("Full UniProt pipeline skipped in offline mode.")

{
  "accessions": "../outputs/uniprot_proteins_300_accessions.txt",
  "fasta": "../outputs/uniprot_proteins_300.fasta",
  "metadata_csv": "../outputs/uniprot_proteins_300_metadata.csv",
  "descriptors_csv": "../outputs/uniprot_proteins_300_descriptors.csv"
}


# 9. Suggested student exercises

## Exercise 1 — FASTA reading

Create a small FASTA file with two DNA sequences and read it using `read_fasta()`.

## Exercise 2 — FASTA export

Create three protein records with:

- sequence ID
- organism
- protein sequence
- description

Export them as FASTA using `write_fasta()`.

## Exercise 3 — UniProt query

Change the UniProt query to retrieve reviewed proteins from another organism.

Examples:

```text
(reviewed:true) AND (organism_id:83333)   # Escherichia coli K-12
(reviewed:true) AND (organism_id:3702)    # Arabidopsis thaliana
(reviewed:true) AND (organism_id:559292)  # Saccharomyces cerevisiae S288C
```

## Exercise 4 — Sequence descriptors

Using the UniProt output, identify:

- the longest protein
- the shortest protein
- the protein with highest hydrophobic fraction
- the protein with highest Shannon entropy

## Exercise 5 — Dataset export

Export a final dataset with these columns:

- `sequence_id`
- `organism`
- `molecule_type`
- `source`
- `sequence`
- `length`
- `hydrophobic_fraction`
- `charged_fraction`
- `shannon_entropy`
- `kmer_entropy_k2`

## Exercise 6 — Prepare for EDA

Save the final table as:

```text
outputs/protein_dataset_for_eda.csv
```

Then load it in a new notebook and inspect:

```python
pd.read_csv("outputs/protein_dataset_for_eda.csv").head()
```

# 10. Conceptual summary

In this notebook, we moved from sequence identifiers to a reusable dataset.

The core pipeline was:

```text
Database query
↓
Sequence IDs
↓
FASTA records
↓
Metadata table
↓
Sequence descriptors
↓
CSV dataset
↓
EDA-ready input
```

This is the practical foundation for the next stage:

```text
EDA of biological sequence datasets
```

In that next notebook, we can analyze:

- length distributions
- descriptor distributions
- organism composition
- redundancy
- sequence diversity
- entropy profiles
- clustering
- distance matrices
- outliers
- dataset quality